[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lorenzo-stacchio/Deep-Learning-and-Computer-Vision-for-Business/blob/main/03-MLLM/vlm_object_detection_ollama.ipynb)

# Object detection with a Vision Language Model: Qwen3.5 + Ollama

A **Vision Language Model (VLM)**, also called a **Multimodal Large Language Model (MLLM)**, is a language model that also accepts images. You can *talk* to it about a picture: describe it, read text, count objects... and even **locate objects**, by asking for their bounding boxes in the answer.

In the detection module we trained **YOLOv12** on the course `retail_products` dataset: a specialised network with a fixed list of classes, trained on hundreds of labelled images. Here we solve the **same task with no training at all**: we write a prompt.

| | YOLOv12 (module 02) | VLM (this notebook) |
|---|---|---|
| Training data | hundreds of annotated images | none (zero-shot) |
| Classes | fixed at training time | whatever you write in the prompt |
| Output | boxes + class scores | text (JSON) that we parse |
| Speed | milliseconds per image | seconds per image |

### The tools
- **[Ollama](https://ollama.com)**: a runtime that downloads and runs open models locally (built on llama.cpp) and exposes them through an HTTP API on port `11434`, with a Python client.
- **[Qwen3.5](https://ollama.com/library/qwen3.5) 4B** (Alibaba): a natively multimodal model. We use the tag `qwen3.5:4b-q4_K_M`: 4 billion parameters **quantized to 4 bits** (`q4_K_M` = llama.cpp "K-quant, medium"). The weights take **3.4 GB** instead of ~9 GB in 16-bit, so the model fits easily on a free Colab T4 GPU (15 GB), with a small loss in accuracy.

### What we will do
1. Check the GPU
2. Install Ollama and start its server
3. Download the quantized Qwen3.5 model
4. Load a sample image (and its ground truth) from the course dataset
5. Chat with the model about the image
6. Prompt it to detect the products: free text vs **structured output**
7. Convert the boxes to pixels and draw them
8. Measure the quality against the ground truth (IoU, precision, recall) on the whole test set
9. Open-vocabulary detection: classes that were never defined

> **Runtime:** in Colab go to `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

### How can a language model output bounding boxes?

A VLM only produces **text tokens**. During training, Qwen models learned to answer grounding questions by *writing* the coordinates of the objects, in a JSON-like format such as:

```json
{"label": "aqua", "bbox_2d": [503, 354, 684, 547]}
```

where `bbox_2d = [x1, y1, x2, y2]` are the top-left and bottom-right corners.

**Important:** Qwen3.5 writes coordinates on a **normalized 0-1000 scale**, independent of the image size: `(0, 0)` is the top-left corner and `(1000, 1000)` the bottom-right one. To draw a box on a `W × H` image we convert:

$$x_{pixel} = \frac{x}{1000} \cdot W \qquad y_{pixel} = \frac{y}{1000} \cdot H$$

(We verified it on this dataset: read as pixels, the boxes are completely wrong; rescaled from 0-1000 they overlap the ground truth.)

## 0. Check the GPU

Ollama automatically uses an NVIDIA GPU if it finds one. On CPU a 4B model also works, but it is much slower (tens of seconds per image).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Install Ollama and start the server

1. `zstd`: the Ollama Linux package is compressed with Zstandard, and the install script stops if `zstd` is missing (it is not installed in Colab by default).
2. The official install script downloads the Ollama binary and its GPU libraries.
3. `ollama`: the Python client for the Ollama API.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

On a normal Linux machine Ollama runs as a system service. Colab has no service manager, so we start `ollama serve` ourselves **as a background process** (its log goes to `ollama.log`) and wait until the API answers.

In [ ]:
import subprocess
import time

import requests

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_is_running():
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.exceptions.RequestException:
        return False


if not ollama_is_running():
    ollama_log = open("ollama.log", "w")
    ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_is_running():
            break
        time.sleep(1)

assert ollama_is_running(), "The Ollama server did not start: check ollama.log"
print("Ollama server version:", requests.get(f"{OLLAMA_URL}/api/version").json()["version"])

## 2. Download the quantized model

`ollama pull` downloads the model once (3.4 GB) into `~/.ollama/models`. Other tags you can try later:

| Tag | Size | Notes |
|---|---|---|
| `qwen3.5:4b-q4_K_M` | 3.4 GB | **used here**: 4-bit quantization |
| `qwen3.5:4b-q8_0` | larger | 8-bit: closer to the original model, slower |
| `qwen3.5:9b` | 6.6 GB | bigger model, usually more accurate |
| `qwen3-vl:4b-instruct-q4_K_M` | 3.3 GB | previous generation, vision-specialised |

In [ ]:
MODEL = "qwen3.5:4b-q4_K_M"

!ollama pull {MODEL}
!ollama list

## 3. Load a sample image from the course dataset

We use the course **`retail_products`** dataset (512×512 images of 6 Indonesian supermarket products, Pascal VOC annotations). The `mix` images of the test split contain **all 6 products together**, so they are a good detection test.

We download the image **and its XML annotation** from the course GitHub repository: the annotation is the *ground truth* we will use to measure the model.

In [ ]:
import os
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

REPO_RAW = "https://raw.githubusercontent.com/lorenzo-stacchio/Deep-Learning-and-Computer-Vision-for-Business/main/02-Pytorch and CV/datasets/retail_products"
DATA_DIR = "retail_products"
CLASSES = ["aqua", "chitato", "indomie", "pepsodent", "shampoo", "tissue"]
COLORS = dict(zip(CLASSES, plt.cm.tab10.colors))


def download(relative_path):
    local_path = os.path.join(DATA_DIR, relative_path)
    if not os.path.exists(local_path):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        url = urllib.parse.quote(f"{REPO_RAW}/{relative_path}", safe=":/")  # spaces and parentheses in file names
        urllib.request.urlretrieve(url, local_path)
    return local_path


def load_ground_truth(annotation_path):
    root = ET.parse(annotation_path).getroot()
    return [{"label": obj.find("name").text,
             "box": [int(obj.find("bndbox").find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]}
            for obj in root.iter("object")]


def draw_boxes(ax, objects, linestyle="-", show_labels=True):
    for obj in objects:
        x1, y1, x2, y2 = obj["box"]
        color = COLORS.get(obj["label"], "red")
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2, linestyle=linestyle))
        if show_labels:
            ax.text(x1, y1 - 4, obj["label"], color="white", fontsize=9, bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor="none"))


IMAGE_NAME = "mix (1)"
image_path = download(f"images/test/{IMAGE_NAME}.jpg")
ground_truth = load_ground_truth(download(f"annotations/test/{IMAGE_NAME}.xml"))
image = Image.open(image_path)
print(f"{image_path}: {image.size[0]}x{image.size[1]} pixels, {len(ground_truth)} annotated objects")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image)
draw_boxes(ax, ground_truth)
ax.set_title("Ground truth (Pascal VOC annotation)")
ax.axis("off")
plt.show()

## 4. First conversation with the model

`ollama.chat()` takes a list of **messages**, like a chat app. An image is attached to a message with the `images` field (a file path, bytes or base64).

Two parameters we will always use:
- `think=False`: Qwen3.5 is a *reasoning* model and by default "thinks" (writes a hidden chain of thought) before answering. For perception tasks like ours this makes answers much slower without a clear benefit, so we disable it.
- `options={"temperature": 0}`: always pick the most likely token, so the same prompt gives the same answer.

The first call also **loads the model into GPU memory**, so it is slower than the next ones.

In [ ]:
import ollama

start = time.time()
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Describe this image in two sentences.", "images": [image_path]}],
    think=False,
    options={"temperature": 0},
)
print(f"({time.time() - start:.1f} s)\n{response.message.content}")

`ollama ps` shows the loaded model and **where it runs**: `100% GPU` means the whole model is in GPU memory.

In [ ]:
!ollama ps

## 5. Ask for bounding boxes

### 5.1 Free-form answer

The simplest approach: describe in the prompt the classes and the output format we want.

In [ ]:
DETECTION_PROMPT = (
    "Detect every retail product in this image. "
    f"The possible classes are: {', '.join(CLASSES)}. "
    'Return a JSON list where each element is {"label": <class>, "bbox_2d": [x1, y1, x2, y2]}.'
)
print(DETECTION_PROMPT, "\n")

start = time.time()
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": DETECTION_PROMPT, "images": [image_path]}],
    think=False,
    options={"temperature": 0},
)
print(f"({time.time() - start:.1f} s)\n{response.message.content}")

The answer *looks* right, but free text is fragile to use in a program:
- the JSON is often wrapped in a markdown code block (```` ```json ````) or surrounded by comments, so we have to extract it with a regular expression;
- nothing guarantees valid JSON, the right keys, or labels from our class list;
- the model can skip objects or invent new labels.

A defensive parser for the free-form answer:

In [ ]:
import json
import re


def parse_free_form(text):
    match = re.search(r"\[.*\]", text, re.DOTALL)  # first [...] block
    if match is None:
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return []


free_form_objects = parse_free_form(response.message.content)
print(f"parsed {len(free_form_objects)} objects:", [obj.get("label") for obj in free_form_objects])

### 5.2 Structured output: force the answer to follow a schema

Ollama supports **structured outputs**: we pass a **JSON schema** in the `format` parameter and Ollama *constrains the generation*, so the model can only produce tokens that respect the schema. The answer is then always valid JSON with the fields we asked for.

We define the schema with **Pydantic**:
- `label` is a `Literal` of our 6 classes: the model cannot invent a new label;
- `bbox_2d` is a list of exactly 4 integers.

Pydantic also validates and parses the answer into Python objects.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Detection(BaseModel):
    label: Literal["aqua", "chitato", "indomie", "pepsodent", "shampoo", "tissue"]
    bbox_2d: list[int] = Field(min_length=4, max_length=4, description="[x1, y1, x2, y2] on a 0-1000 scale")


class Detections(BaseModel):
    objects: list[Detection]


print(json.dumps(Detections.model_json_schema(), indent=2))

In [ ]:
start = time.time()
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": DETECTION_PROMPT, "images": [image_path]}],
    format=Detections.model_json_schema(),
    think=False,
    options={"temperature": 0},
)
print(f"({time.time() - start:.1f} s)\n{response.message.content}\n")

detections = Detections.model_validate_json(response.message.content)
for obj in detections.objects:
    print(obj)

## 6. From the 0-1000 scale to pixels

We convert each box to pixels, clip it inside the image and make sure that `x1 < x2` and `y1 < y2` (a model can occasionally swap the corners). Then we draw the predictions (**solid**) on top of the ground truth (**dashed**).

In [ ]:
def to_pixels(bbox_2d, width, height):
    x1, x2 = sorted(min(max(v, 0), 1000) / 1000 * width for v in (bbox_2d[0], bbox_2d[2]))
    y1, y2 = sorted(min(max(v, 0), 1000) / 1000 * height for v in (bbox_2d[1], bbox_2d[3]))
    return [x1, y1, x2, y2]


def detect_products(image_path, prompt=DETECTION_PROMPT, schema=Detections):
    width, height = Image.open(image_path).size
    start = time.time()
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt, "images": [image_path]}],
        format=schema.model_json_schema(),
        think=False,
        options={"temperature": 0},
    )
    elapsed = time.time() - start
    objects = schema.model_validate_json(response.message.content).objects
    return [{"label": obj.label, "box": to_pixels(obj.bbox_2d, width, height)} for obj in objects], elapsed


predictions, elapsed = detect_products(image_path)
print(f"{len(predictions)} objects in {elapsed:.1f} s")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image)
draw_boxes(ax, ground_truth, linestyle="--", show_labels=False)
draw_boxes(ax, predictions)
ax.set_title(f"{MODEL}: predictions (solid) vs ground truth (dashed)")
ax.axis("off")
plt.show()

## 7. How good are the boxes?

We compare predictions and ground truth with the **Intersection over Union (IoU)**: the area where the two boxes overlap divided by the area they cover together. `IoU = 1` is a perfect box, `IoU = 0` means no overlap.

For each class we match predicted and ground-truth boxes one-to-one with the Hungarian algorithm (maximising the total IoU). As in the standard detection metrics (e.g. mAP@0.5), a prediction is a **true positive (TP)** if its IoU with the matched ground-truth box is **≥ 0.5**; unmatched predictions are **false positives (FP)** and missed objects **false negatives (FN)**.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

IOU_THRESHOLD = 0.5


def box_iou(a, b):
    inter_w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_detections(predictions, ground_truth, iou_threshold=IOU_THRESHOLD):
    rows, n_false_positives = [], 0
    for label in sorted({o["label"] for o in predictions + ground_truth}):
        preds = [p["box"] for p in predictions if p["label"] == label]
        gts = [g["box"] for g in ground_truth if g["label"] == label]
        ious = np.array([[box_iou(p, g) for g in gts] for p in preds]).reshape(len(preds), len(gts))
        matched = {}
        if preds and gts:
            for p_idx, g_idx in zip(*linear_sum_assignment(-ious)):
                if ious[p_idx, g_idx] >= iou_threshold:
                    matched[g_idx] = ious[p_idx, g_idx]
        n_false_positives += len(preds) - len(matched)
        for g_idx in range(len(gts)):
            best_iou = ious[:, g_idx].max() if preds else 0.0
            rows.append({"label": label, "iou": matched.get(g_idx, best_iou), "detected": g_idx in matched})
    return pd.DataFrame(rows), n_false_positives


matches, n_false_positives = match_detections(predictions, ground_truth)
print(f"TP: {matches['detected'].sum()} | FN: {(~matches['detected']).sum()} | FP: {n_false_positives}")
matches.round(2)

### 7.1 Evaluation on the whole test set

One image is not enough to judge a model. The test split contains **26 `mix` images** (each with the 6 products): we run the same prompt on all of them and compute
- **precision** = TP / (TP + FP): how many predicted boxes are correct;
- **recall** = TP / (TP + FN): how many products were found;
- **mean IoU** of the correct boxes: how precise the localisation is;
- the **average time per image**.

On a T4 GPU this takes a couple of minutes.

In [ ]:
N_EVAL_IMAGES = 26
eval_names = [f"mix ({i})" for i in range(1, N_EVAL_IMAGES + 1)]

all_matches, results = [], []
for name in eval_names:
    path = download(f"images/test/{name}.jpg")
    gt = load_ground_truth(download(f"annotations/test/{name}.xml"))
    preds, elapsed = detect_products(path)
    m, fp = match_detections(preds, gt)
    all_matches.append(m.assign(image=name))
    results.append({"image": name, "path": path, "gt": gt, "preds": preds, "tp": int(m["detected"].sum()),
                    "fn": int((~m["detected"]).sum()), "fp": fp, "seconds": elapsed})
    print(f"{name:>9}: {len(preds)} boxes, TP={results[-1]['tp']} FN={results[-1]['fn']} FP={fp} ({elapsed:.1f} s)")

all_matches = pd.concat(all_matches, ignore_index=True)
summary = pd.DataFrame(results)
tp, fp, fn = summary["tp"].sum(), summary["fp"].sum(), summary["fn"].sum()
print(f"\nprecision: {tp / (tp + fp):.2%} | recall: {tp / (tp + fn):.2%} | "
      f"mean IoU of correct boxes: {all_matches.loc[all_matches['detected'], 'iou'].mean():.2f} | "
      f"average time: {summary['seconds'].mean():.1f} s/image")

In [ ]:
per_class = all_matches.groupby("label").agg(recall=("detected", "mean"), mean_iou=("iou", "mean"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
per_class["recall"].plot.bar(ax=axes[0], color=[COLORS[c] for c in per_class.index], ylim=(0, 1.05),
                             title=f"Recall per class (IoU >= {IOU_THRESHOLD})")
axes[1].hist(all_matches["iou"], bins=20, range=(0, 1))
axes[1].axvline(IOU_THRESHOLD, color="red", linestyle="--")
axes[1].set_title("IoU of every ground-truth object with its best prediction")
plt.tight_layout()
plt.show()
per_class.round(2)

Let's look at the images where the model made the most mistakes.

In [ ]:
worst = summary.assign(errors=summary["fn"] + summary["fp"]).sort_values("errors", ascending=False).head(4)

fig, axes = plt.subplots(1, len(worst), figsize=(5 * len(worst), 5))
for ax, (_, row) in zip(np.atleast_1d(axes), worst.iterrows()):
    ax.imshow(Image.open(row["path"]))
    draw_boxes(ax, row["gt"], linestyle="--", show_labels=False)
    draw_boxes(ax, row["preds"])
    ax.set_title(f"{row['image']}: TP={row['tp']} FN={row['fn']} FP={row['fp']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Open-vocabulary detection

The real strength of a VLM is that **the classes are just words in the prompt**. We can ask for objects that YOLO was never trained on, or for a richer description, *without any new data or training*.

Here we remove the class list: the model must describe each package with its own words. The schema now allows any string as `label`, so we can no longer compute per-class metrics; we only check **how many annotated products received a box** (IoU ≥ 0.5, whatever the label).

Expect this to be **less reliable** than detection with a class list: without a closed vocabulary the model may skip packages, return the same product twice, use vague labels ("product package") or even invent brand names. The class list in the prompt and the `Literal` in the schema are what kept it focused in the previous sections.

In [ ]:
class OpenDetection(BaseModel):
    label: str = Field(description="short description: brand and product type")
    bbox_2d: list[int] = Field(min_length=4, max_length=4, description="[x1, y1, x2, y2] on a 0-1000 scale")


class OpenDetections(BaseModel):
    objects: list[OpenDetection]


OPEN_PROMPT = ("There are several product packages in this image. Return one entry for EACH package "
               "(bottles, bags, boxes, packs), with a short label describing the product type "
               "and its bbox_2d [x1, y1, x2, y2].")

open_predictions, elapsed = detect_products(image_path, prompt=OPEN_PROMPT, schema=OpenDetections)
print(f"({elapsed:.1f} s)")
for obj in open_predictions:
    print(f"{obj['label']:<35} {[round(v) for v in obj['box']]}")

covered = [max((box_iou(p["box"], gt["box"]) for p in open_predictions), default=0) >= IOU_THRESHOLD for gt in ground_truth]
print(f"\nannotated products with a box (IoU >= {IOU_THRESHOLD}): {sum(covered)}/{len(ground_truth)}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image)
draw_boxes(ax, open_predictions)
ax.set_title("Open-vocabulary detection (no class list)")
ax.axis("off")
plt.show()

## 9. Discussion

**When is a VLM a good detector?**
- **No labelled data yet**: get a first detector in minutes, or pre-annotate a dataset and only *correct* the boxes (as we did with Grounding DINO + SAM in the segmentation module).
- **Long tail / changing catalogue**: new products only require changing the prompt.
- **Richer outputs**: in the same answer the model can also read the brand, the price tag or the packaging state.

**Limitations**
- **Speed and cost**: seconds per image on a GPU vs milliseconds for YOLO. Not suitable for real-time video.
- **Localisation precision**: coordinates are written as tokens on a 0-1000 grid, and a 4B quantized model is less precise than a detector trained on the domain.
- **Hallucinations**: the model can miss objects, duplicate them or label them wrongly; structured outputs fix the *format*, not the *content*.
- **Prompt sensitivity**: small changes to the prompt can change the results. Always evaluate on labelled data, as we did.

**Practical tips**: `temperature=0`, structured outputs with a strict schema, `think=False` for perception tasks, and always convert coordinates with the convention of the model you use (Qwen3.5 and Qwen3-VL: 0-1000; other VLMs use pixels or 0-1).

### Exercises
1. Replace `MODEL` with `qwen3.5:4b-q8_0` or `qwen3.5:9b` and compare precision, recall, mean IoU and time per image.
2. Try `qwen3-vl:4b-instruct-q4_K_M`: is the previous, vision-specialised generation better at localisation?
3. Enable thinking (`think=True`): does reasoning improve the boxes? How much slower is it?
4. Run the model on the **single-product** test images (`aqua (1)`, ...) and compute the classification accuracy.
5. Compare the metrics with the YOLOv12 model fine-tuned in `02-Pytorch and CV/02_detection/yolov12_retail_products`.
6. Use the model to pre-annotate the `train` split and save the predictions as Pascal VOC XML files.